In [152]:
import pandas as pd
import numpy as np
orders = pd.read_csv('../csv-files/olist_orders_dataset.csv')


Phase 1: Structural and schema check

In [154]:
#column check
expected_columns = [
    'order_id', 'customer_id', 'order_status',
    'order_purchase_timestamp', 'order_approved_at',
    'order_delivered_carrier_date', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

actual_columns = list(orders.columns)

missing = [c for c in expected_columns if c not in actual_columns]
extra = [c for c in actual_columns if c not in expected_columns]

print("Missing columns:", missing)
print("Unexpected extra columns:", extra)

Missing columns: []
Unexpected extra columns: []


In [155]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


Phase 2: duplicates check

In [156]:
full_dupes = orders.duplicated()
print("Full-row duplicates found:", full_dupes.sum())

Full-row duplicates found: 0


In [157]:
# checking duplicate key columns like order_id that should be unique
dupe_ids = orders['order_id'].duplicated()
print("Duplicate order_id count:", dupe_ids.sum())

Duplicate order_id count: 0


Phase 3: checking if everything that needs to be unique is unique

In [158]:
print("Is order_id fully unique?", orders['order_id'].is_unique)
print("Total rows:", len(orders))
print("Unique order_id count:", orders['order_id'].nunique())

Is order_id fully unique? True
Total rows: 99441
Unique order_id count: 99441


Phase 4: Setting up correct data types

In [159]:
# using orders.info to check data type of each column we see dates are also
# has the data type string
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [160]:
# first we create a varriable that contains every column that needs data type change
date_cols = ['order_purchase_timestamp', 'order_approved_at', 
             'order_delivered_carrier_date', 'order_delivered_customer_date', 
             'order_estimated_delivery_date']

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [161]:
# renamed somme of the columns, because some of them are too long we renamed them
# with column names that still represent them properly for readablity 
orders = orders.rename(columns={
    'order_purchase_timestamp': 'purchase_ts',
    'order_approved_at': 'approved_ts',
    'order_delivered_carrier_date': 'carrier_delivered_dt',
    'order_delivered_customer_date': 'customer_delivered_dt',
    'order_estimated_delivery_date': 'est_delivery_dt',
})

Phase 5: 

In [162]:
orders['order_status'].apply(lambda x: x != x.strip().lower()).sum()

np.int64(0)

In [163]:
# use this code to normalize if there is any casing or stray whitespace
orders['order_status'] = orders['order_status'].str.strip().str.lower()

In [164]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [165]:
orders['order_status'] = orders['order_status'].replace('canceled', 'cancelled')

In [166]:
known_statuses = ['delivered', 'shipped', 'cancelled', 'unavailable',
                  'invoiced', 'processing', 'created', 'approved']

current_values = orders['order_status'].unique()
unexpected = [v for v in current_values if v not in known_statuses]
print("Unexpected status values:", unexpected)

Unexpected status values: []


In [167]:
orders['order_status'].nunique()

8

In [168]:
int(orders['order_status'].isnull().sum())

0

Phase 6:

In [169]:
for col in ['purchase_ts', 'approved_ts', 
        'carrier_delivered_dt', 'customer_delivered_dt', 
        'est_delivery_dt']:
    print(col, "- NaT count:", orders[col].isnull().sum())

purchase_ts - NaT count: 0
approved_ts - NaT count: 160
carrier_delivered_dt - NaT count: 1783
customer_delivered_dt - NaT count: 2965
est_delivery_dt - NaT count: 0


In [170]:
cancelled = orders[orders['carrier_delivered_dt'].isnull()]
cancelled['order_status'].value_counts()

order_status
unavailable    609
cancelled      550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

In [172]:
cancelled[cancelled['order_status'] == "delivered"]

,order_id,customer_id,order_status,purchase_ts,approved_ts,carrier_delivered_dt,customer_delivered_dt,est_delivery_dt
73222,2aa91108853cecb43c84a5dc5b277475,afeb16c7f46396c0ed54acb45ccaaa40,delivered,2017-09-29 08:52:58,2017-09-29 09:07:16,NaT,2017-11-20 19:44:47,2017-11-14
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23


unavailable - purchased - approved - n.carr.dlv.dt  - n.cust_dlv.dt - est.dlv.dt = this is good total of unavailable: 609
cancelled - purchase - can be approved or n.approved - n.carr.dlv.dt  - n.cust_dlv.dt = this is good total of cancelled: 550
invoiced - purchase - approved - n.carr.dlv.dt  - n.cust_dlv.dt - est.dlv.dt = this is good total of invoiced: 314
processing - purchased - approved - n.carr.dlv.dt  - n.cust_dlv.dt - est.dlv.dt = this is good total of processing: 301
created - purchased - n.approved - n.carr.dlv.dt  - n.cust_dlv.dt - est.dlv.dt = this is good total of created: 5
approved - purchased - approved - n.carr.dlv.dt  - n.cust_dlv.dt - est.dlv.dt = this is good total of approved: 2
delivered - purchased - approved - n.carr.dlv.dt  - cust_dlv.dt - est.dlv.dt = this is good total of delivered: 2



In [173]:
orders[orders['order_status'] == "approved"].head()

,order_id,customer_id,order_status,purchase_ts,approved_ts,carrier_delivered_dt,customer_delivered_dt,est_delivery_dt
44897,a2e4c44360b4a57bdff22f3a4630c173,8886130db0ea6e9e70ba0b03d7c0d286,approved,2017-02-06 20:18:17,2017-02-06 20:30:19,NaT,NaT,2017-03-01
88457,132f1e724165a07f6362532bfb97486e,b2191912d8ad6eac2e4dc3b6e1459515,approved,2017-04-25 01:25:34,2017-04-30 20:32:41,NaT,NaT,2017-05-22


In [174]:
mask = (
    (orders['order_status'] == 'delivered') &
    (orders['approved_ts'].isnull()) 
)

orders.loc[mask, 'approved_ts'] = orders.loc[mask, 'purchase_ts']

In [175]:
conditions = [
    # condition 1: delivered — needs all 5 dates
    (orders['order_status'] == 'delivered') &
    orders['purchase_ts'].notnull() &
    orders['approved_ts'].notnull() &
    orders['carrier_delivered_dt'].notnull() &
    orders['customer_delivered_dt'].notnull() &
    orders['est_delivery_dt'].notnull(),

    # condition 2: cancelled — maybe only needs purchase date, no delivery dates
    (orders['order_status'] == 'cancelled') &
    orders['purchase_ts'].notnull() &
    orders['customer_delivered_dt'].isnull(),

    # condition 3: invoiced — needs purchase + approved, but not delivered yet
    (orders['order_status'] == 'invoiced') &
    orders['purchase_ts'].notnull() &
    orders['approved_ts'].notnull(),

    # condition 4: shipped — has carrier date but not customer date
    (orders['order_status'] == 'shipped') &
    orders['purchase_ts'].notnull() &
    orders['approved_ts'].notnull() &
    orders['carrier_delivered_dt'].notnull() &
    orders['customer_delivered_dt'].isnull() &
    orders['est_delivery_dt'].notnull(),

    #5
    (orders['order_status'] == 'unavailable') &
    orders['purchase_ts'].notnull() &
    orders['approved_ts'].notnull()&
    orders['carrier_delivered_dt'].isnull() &
    orders['customer_delivered_dt'].isnull(),

    #6
    (orders['order_status'] == 'processing') &
    orders['purchase_ts'].notnull() &
    orders['approved_ts'].notnull()&
    orders['carrier_delivered_dt'].isnull() &
    orders['customer_delivered_dt'].isnull(),

    #7
    (orders['order_status'] == 'created') &
    orders['purchase_ts'].notnull() &
    orders['approved_ts'].isnull()&
    orders['carrier_delivered_dt'].isnull() &
    orders['customer_delivered_dt'].isnull(),

    #8
    (orders['order_status'] == 'approved') &
    orders['purchase_ts'].notnull() &
    orders['approved_ts'].notnull()&
    orders['carrier_delivered_dt'].isnull() &
    orders['customer_delivered_dt'].isnull(),

    #9 cancelled but delivered
    (orders['order_status'] == 'cancelled') &
    orders['purchase_ts'].notnull() &
    orders['approved_ts'].notnull()&
    orders['carrier_delivered_dt'].notnull() &
    orders['customer_delivered_dt'].notnull(),

    #10 delivered - no delivery date
    (orders['order_status'] == 'delivered') &
    orders['purchase_ts'].notnull() &
    orders['approved_ts'].notnull() &
    orders['carrier_delivered_dt'].notnull() &
    orders['customer_delivered_dt'].isnull() &
    orders['est_delivery_dt'].notnull(),

    #11 delivered — courrier delivery date
    (orders['order_status'] == 'delivered') &
    orders['purchase_ts'].notnull() &
    orders['approved_ts'].notnull() &
    orders['carrier_delivered_dt'].isnull() &
    orders['customer_delivered_dt'].notnull() &
    orders['est_delivery_dt'].notnull(),

    #12 delivered — no courrier and delivery date
    (orders['order_status'] == 'delivered') &
    orders['purchase_ts'].notnull() &
    orders['approved_ts'].notnull() &
    orders['carrier_delivered_dt'].isnull() &
    orders['customer_delivered_dt'].isnull() &
    orders['est_delivery_dt'].notnull()


]

choices = [
    'good',
    'good',
    'good',
    'good',
    'good',
    'good',
    'good',
    'good',
    'delivered then cancelled',
    'delivered no delivery_dt',
    'delivered no carrier_dt',
    'delivered no carrier & delivery_dt'
]

orders['status_check'] = np.select(conditions, choices, default='bad')

In [176]:
order_checker = orders[orders['status_check'] == "bad"]
order_checker['order_status'].value_counts()

Series([], Name: count, dtype: int64)

In [177]:
# to use this to check update the status_check above with the proper remarks
order_checker[order_checker['order_status'] == "delivered"].head(20)

,order_id,customer_id,order_status,purchase_ts,approved_ts,carrier_delivered_dt,customer_delivered_dt,est_delivery_dt,status_check


In [178]:
# approved before purchase

order_dt_check = orders[orders['status_check'] == "good"]

bad_approval = order_dt_check[order_dt_check['approved_ts'] < order_dt_check['purchase_ts']]
print("Approved before purchase:", len(bad_approval))

# delivered to carrier before approved
bad_carrier = order_dt_check[order_dt_check['carrier_delivered_dt'] < order_dt_check['approved_ts']]
print("Delivered to carrier before approval:", len(bad_carrier))

# delivered to customer before delivered to carrier
bad_customer_delivery = order_dt_check[order_dt_check['customer_delivered_dt'] < order_dt_check['carrier_delivered_dt']]
print("Delivered to customer before carrier handoff:", len(bad_customer_delivery))

Approved before purchase: 0
Delivered to carrier before approval: 1359
Delivered to customer before carrier handoff: 23


In [179]:
# we can really change the date for this orders because we dont have any information that could help us correct this
# we could try to use average for this but only if we really need to
conditions = [
    orders['carrier_delivered_dt'] < orders['approved_ts'],
    orders['customer_delivered_dt'] < orders['carrier_delivered_dt'],
]

choices = [
    'approved_carrier_mismatch',
    'customer_before_carrier_mismatch',
]

orders['sequence_flag'] = np.select(conditions, choices, default='ok')

In [149]:
count = bad_carrier['carrier_delivered_dt'].notnull()
count.sum()

np.int64(1359)

In [181]:
count = bad_customer_delivery['customer_delivered_dt'].isnull()
count.sum()

np.int64(0)

In [180]:
# checking if there is any overalppp between mismatches, if there is
# a cas that both conditions are met in a row it will choose the first choice
# rather than flag both
count = bad_customer_delivery['order_id'].isin(bad_carrier['order_id'])
count.sum()

np.int64(0)

In [182]:
bad_carrier['time_diff'] = (bad_carrier['approved_ts'] - bad_carrier['carrier_delivered_dt'])
bad_carrier['time_diff'].describe()

count                      1359
mean     1 days 00:45:07.153053
std      4 days 19:18:28.055754
min             0 days 00:00:21
25%      0 days 01:24:55.500000
50%             0 days 17:10:04
75%             1 days 01:57:24
max           171 days 05:15:22
Name: time_diff, dtype: object

In [183]:
bad_customer_delivery['time_diff'] = (bad_customer_delivery['carrier_delivered_dt'] - bad_customer_delivery['customer_delivered_dt'])
bad_customer_delivery['time_diff'].describe()

count                        23
mean     3 days 06:27:18.478260
std      3 days 17:18:29.050924
min             0 days 00:23:18
25%             0 days 23:20:18
50%             1 days 15:51:52
75%      5 days 08:45:32.500000
max            16 days 02:18:29
Name: time_diff, dtype: object

In [184]:
#sum count of null in each columns
orders.isnull().sum()

order_id                    0
customer_id                 0
order_status                0
purchase_ts                 0
approved_ts               146
carrier_delivered_dt     1783
customer_delivered_dt    2965
est_delivery_dt             0
status_check                0
sequence_flag               0
dtype: int64

In [187]:
orders['delivery_delay_days'] = np.ceil(
    (orders['customer_delivered_dt'] - orders['est_delivery_dt']).dt.total_seconds() / 86400
)

In [188]:
orders.head()

,order_id,customer_id,order_status,purchase_ts,approved_ts,carrier_delivered_dt,customer_delivered_dt,est_delivery_dt,status_check,sequence_flag,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,good,ok,-7.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,good,ok,-5.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,good,ok,-17.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,good,ok,-12.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,good,ok,-9.0


In [189]:
orders.to_csv('../df_cleaned/orders_info_cln.csv')